# OpenRouter free-model financial-data test

Compare currently available free OpenRouter models on a small, local financial dataset. The notebook sends only the rows shown in `data_preview`; review them before running model calls. Free-model availability and rate limits can change.

## Setup

1. Create an API key at [OpenRouter](https://openrouter.ai/keys).
2. Add `OPENROUTER_API_KEY=...` to the project-root `.env` file.
3. Select this project's Python environment as the notebook kernel.

Do not paste keys into notebook cells or commit `.env`.

In [ ]:
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Works whether Jupyter starts in the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / '.env')

API_KEY = os.getenv('OPENROUTER_API_KEY', '').strip()
BASE_URL = 'https://openrouter.ai/api/v1'
HEADERS = {
    'Authorization': f'Bearer {API_KEY}',
    'Content-Type': 'application/json',
    'HTTP-Referer': 'http://localhost',
    'X-OpenRouter-Title': 'Financial Analyst Model Test',
}
assert API_KEY, 'Add OPENROUTER_API_KEY to the project-root .env file, then rerun this cell.'
print('OpenRouter key loaded (value hidden).')

: 

## Load a small financial-data sample

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'sample_holdings.csv'  # Change to another project file if desired.
MAX_ROWS_SENT = 25

def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.json':
        return pd.read_json(path)
    raise ValueError(f'Use a CSV, XLSX, XLS, or JSON file; got {suffix!r}.')

financial_data = load_table(DATA_PATH)
data_preview = financial_data.head(MAX_ROWS_SENT).copy()
print(f'Loaded {len(financial_data):,} rows from {DATA_PATH.relative_to(PROJECT_ROOT)}; {len(data_preview)} will be sent.')
display(data_preview)

Before continuing, confirm the preview contains no account numbers, client identifiers, material non-public information, or other data you should not send to a third party.

## Discover today's free text models

In [ ]:
def get_free_models() -> pd.DataFrame:
    response = requests.get(f'{BASE_URL}/models', headers=HEADERS, timeout=30)
    response.raise_for_status()
    rows = []
    for model in response.json().get('data', []):
        pricing = model.get('pricing') or {}
        architecture = model.get('architecture') or {}
        try:
            is_free = float(pricing.get('prompt', 1)) == 0 and float(pricing.get('completion', 1)) == 0
        except (TypeError, ValueError):
            is_free = False
        inputs = architecture.get('input_modalities') or []
        outputs = architecture.get('output_modalities') or []
        if is_free and 'text' in inputs and 'text' in outputs:
            rows.append({
                'id': model['id'],
                'name': model.get('name', model['id']),
                'context_length': model.get('context_length'),
                'parameters': ', '.join(model.get('supported_parameters') or []),
            })
    return pd.DataFrame(rows).sort_values(['context_length', 'name'], ascending=[False, True]).reset_index(drop=True)

free_models = get_free_models()
print(f'Found {len(free_models)} zero-priced text models. This list is live and can change.')
display(free_models)

## Configure a fair comparison

In [ ]:
# Start with the free router. To compare named models, copy up to 3 IDs from the table above.
MODELS_TO_TEST = ['openrouter/free']
MAX_OUTPUT_TOKENS = 700
TEMPERATURE = 0.1

SYSTEM_PROMPT = '''You are a careful financial-data analyst. Use only the supplied data.
Separate facts from interpretations, show arithmetic, flag missing fields, and do not invent prices,
returns, fundamentals, or recommendations. This is a model evaluation, not investment advice.'''

TASK = '''Analyze this dataset. Return these sections:
1. Data-quality observations
2. Three useful calculations, with formulas and results
3. Concentration or risk observations justified by the available columns
4. Missing data needed for a stronger portfolio analysis
5. A concise bottom line
Do not treat cost basis as current market value.'''

DATA_TEXT = data_preview.to_csv(index=False)
USER_PROMPT = f'{TASK}\n\nDataset ({DATA_PATH.name}):\n```csv\n{DATA_TEXT}```'
print(USER_PROMPT)

In [ ]:
def test_model(model_id: str) -> dict:
    payload = {
        'model': model_id,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': USER_PROMPT},
        ],
        'temperature': TEMPERATURE,
        'max_tokens': MAX_OUTPUT_TOKENS,
    }
    started = time.perf_counter()
    try:
        response = requests.post(f'{BASE_URL}/chat/completions', headers=HEADERS, json=payload, timeout=120)
        latency = time.perf_counter() - started
        response.raise_for_status()
        body = response.json()
        message = body['choices'][0]['message']
        usage = body.get('usage') or {}
        return {
            'requested_model': model_id,
            'served_model': body.get('model', model_id),
            'status': 'ok',
            'latency_seconds': round(latency, 2),
            'prompt_tokens': usage.get('prompt_tokens'),
            'completion_tokens': usage.get('completion_tokens'),
            'response': message.get('content') or message.get('reasoning') or '',
            'error': '',
        }
    except (requests.RequestException, KeyError, IndexError, ValueError) as exc:
        detail = ''
        if 'response' in locals():
            try:
                detail = response.json().get('error', {}).get('message', response.text[:500])
            except ValueError:
                detail = response.text[:500]
        return {
            'requested_model': model_id, 'served_model': '', 'status': 'error',
            'latency_seconds': round(time.perf_counter() - started, 2),
            'prompt_tokens': None, 'completion_tokens': None, 'response': '',
            'error': detail or str(exc),
        }

## Run the comparison

Free endpoints are rate-limited and can be temporarily unavailable. This runs sequentially and pauses briefly between calls. Keep the test list small.

In [ ]:
if len(MODELS_TO_TEST) > 3:
    raise ValueError('Keep MODELS_TO_TEST to 3 or fewer to respect free-tier capacity.')

results = []
for index, model_id in enumerate(MODELS_TO_TEST):
    print(f'Testing {model_id} ...')
    results.append(test_model(model_id))
    if index < len(MODELS_TO_TEST) - 1:
        time.sleep(2)

summary = pd.DataFrame(results).drop(columns=['response'])
display(summary)
for result in results:
    display(Markdown(f"### {result['requested_model']} → {result['served_model'] or 'failed'}"))
    display(Markdown(result['response'] or f"**Error:** {result['error']}"))

## Score the outputs yourself

Use a 1–5 scale for correctness, instruction-following, numerical accuracy, clarity, and appropriate uncertainty. Human review matters: speed and fluent prose do not establish financial correctness.

In [ ]:
scorecard = pd.DataFrame({
    'model': [r['served_model'] or r['requested_model'] for r in results],
    'correctness_1_to_5': [None] * len(results),
    'numeric_accuracy_1_to_5': [None] * len(results),
    'instruction_following_1_to_5': [None] * len(results),
    'clarity_1_to_5': [None] * len(results),
    'uncertainty_1_to_5': [None] * len(results),
    'notes': [''] * len(results),
})
display(scorecard)

# Optional: uncomment after filling scores to preserve a local comparison.
# scorecard.to_csv(PROJECT_ROOT / 'data' / 'openrouter_model_scorecard.csv', index=False)